# Titanic: Data Cleaning & Feature Engineering

This notebook loads the Kaggle Titanic dataset, engineers several useful features
(Name title, deck, ticket prefix, family size, etc.), and saves a processed
version for modeling.

**Contents**
1. Setup & paths  
2. Load raw data  
3. Quick peek  
4. Helper extractors (title/deck/ticket prefix)  
5. Cleaning function  
6. Clean train/test  
7. Save processed files  
8. Sanity checks & column order

In [17]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

train_path = RAW / "train.csv"
test_path  = RAW / "test.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

train.head(), train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


(   PassengerId  Survived  Pclass  \
 0            1         0       3   
 1            2         1       1   
 2            3         1       3   
 3            4         1       1   
 4            5         0       3   
 
                                                 Name     Sex   Age  SibSp  \
 0                            Braund, Mr. Owen Harris    male  22.0      1   
 1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
 2                             Heikkinen, Miss. Laina  female  26.0      0   
 3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
 4                           Allen, Mr. William Henry    male  35.0      0   
 
    Parch            Ticket     Fare Cabin Embarked  
 0      0         A/5 21171   7.2500   NaN        S  
 1      0          PC 17599  71.2833   C85        C  
 2      0  STON/O2. 3101282   7.9250   NaN        S  
 3      0            113803  53.1000  C123        S  
 4      0            373450   8.0500

## Quick Peek at Raw Data
We inspect the head and basic info to confirm types and missingness.

In [18]:
# Show sample rows
display(train.head())

# Info summary (types + non-null counts)
train.info()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## Helper Extractors
Functions to parse:
- **Title** from `Name`
- **Deck** from `Cabin`
- **TicketPrefix** from `Ticket`


In [19]:
import re
import pandas as pd

def extract_title(name: str) -> str:
    # Extract title between comma and dot; e.g., "Braund, Mr. Owen Harris" -> "Mr"
    m = re.search(r",\s*([^\.]+)\.", name)
    title = m.group(1).strip() if m else "Unknown"

    # Map rare titles to a single "Rare" group
    rare = {"Lady","Countess","Capt","Col","Don","Dr","Major","Rev","Sir","Jonkheer","Dona"}
    title = "Rare" if title in rare else title

    # Normalize common variants
    title = title.replace("Mme","Mrs").replace("Mlle","Miss").replace("Ms","Miss")
    return title

def extract_deck(cabin: str) -> str:
    # Take the first letter of cabin; missing -> "U" (Unknown)
    if pd.isna(cabin) or not isinstance(cabin, str) or not cabin.strip():
        return "U"
    return cabin.strip()[0]

def ticket_prefix(ticket: str) -> str:
    # Keep non-numeric prefix to reduce cardinality (e.g., "PC", "STON", "A/5")
    if pd.isna(ticket):
        return "UNK"
    t = ticket.replace(".", "").replace("/", "").strip()

    # If pure numeric after cleaning, return "NUM"
    if t.replace(" ", "").isdigit():
        return "NUM"

    # Keep only non-digits as the prefix
    pref = re.sub(r"[0-9\s]+", "", t)
    return pref if pref else "NUM"


## Cleaning Function
- Ensures types  
- Imputes `Embarked` with mode  
- Fills `Fare` per `Pclass` median  
- Engineers: `Title`, `FamilySize`, `IsAlone`, `Deck`, `HasCabin`, `TicketGroupSize`, `TicketPrefix`  
- Drops high-cardinality originals (`Name`, `Ticket`, `Cabin`)  
- Reorders columns for readability  
> Note: **Do not impute `Age` here**; handle in modeling pipeline (e.g., `SimpleImputer`).


In [20]:
def clean_titanic(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ensure types
    if "Pclass" in df.columns:
        df["Pclass"] = df["Pclass"].astype("int64")

    # Embarked: fill missing with mode
    if "Embarked" in df.columns:
        mode_emb = df["Embarked"].mode(dropna=True)
        if not mode_emb.empty:
            df["Embarked"] = df["Embarked"].fillna(mode_emb.iloc[0])

    # Fare: fill missing with class-wise median
    if "Fare" in df.columns:
        df["Fare"] = df.groupby("Pclass")["Fare"].transform(
            lambda s: s.fillna(s.median())
        )

    # --- Engineered features ---
    # 1) Title from Name
    df["Title"] = df["Name"].apply(extract_title) if "Name" in df.columns else "Unknown"

    # 2) FamilySize & IsAlone
    df["FamilySize"] = df[["SibSp", "Parch"]].sum(axis=1) + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # 3) Deck & HasCabin from Cabin
    df["Deck"] = df["Cabin"].apply(extract_deck) if "Cabin" in df.columns else "U"
    df["HasCabin"] = df["Cabin"].notna().astype(int) if "Cabin" in df.columns else 0

    # 4) Ticket group size & prefix
    if "Ticket" in df.columns:
        grp_counts = df["Ticket"].value_counts()
        df["TicketGroupSize"] = df["Ticket"].map(grp_counts).fillna(1).astype(int)
        df["TicketPrefix"] = df["Ticket"].apply(ticket_prefix)
    else:
        df["TicketGroupSize"] = 1
        df["TicketPrefix"] = "UNK"

    # Important: do NOT impute Age here (leave for model pipeline)
    # Drop high-cardinality originals replaced by engineered features
    drop_cols = [c for c in ["Name", "Ticket", "Cabin"] if c in df.columns]
    df = df.drop(columns=drop_cols)

    # Preferred column order for readability
    preferred_order = [
        "Survived","Pclass","Sex","Age","SibSp","Parch","FamilySize","IsAlone",
        "Fare","Embarked","Deck","HasCabin","TicketGroupSize","TicketPrefix","Title",
        "PassengerId"
    ]
    cols = [c for c in preferred_order if c in df.columns] + [c for c in df.columns if c not in preferred_order]
    df = df[cols]

    return df


## Clean Train/Test
Run the cleaning function and preview results.

In [ ]:
train_clean = clean_titanic(train)
test_clean  = clean_titanic(test)

# Peek at cleaned training data
display(train_clean.head())

# Top missing columns (sanity)
display(
    train_clean.isna().sum().sort_values(ascending=False).head(10)
)

,Survived,Pclass,Sex,Age,SibSp,Parch,FamilySize,IsAlone,Fare,Embarked,Deck,HasCabin,TicketGroupSize,TicketPrefix,Title,PassengerId
0,0,3,male,22.0,1,0,2,0,7.2500,S,U,0,1,A,Mr,1
1,1,1,female,38.0,1,0,2,0,71.2833,C,C,1,1,PC,Mrs,2
2,1,3,female,26.0,0,0,1,1,7.9250,S,U,0,1,STONO,Miss,3
3,1,1,female,35.0,1,0,2,0,53.1000,S,C,1,2,NUM,Mrs,4
4,0,3,male,35.0,0,0,1,1,8.0500,S,U,0,1,NUM,Mr,5


Age           177
Survived        0
Pclass          0
Sex             0
SibSp           0
Parch           0
FamilySize      0
IsAlone         0
Fare            0
Embarked        0
dtype: int64

## Save Processed Datasets
Write clean CSVs to `data/processed/`.


In [22]:
train_out = PROC / "train_clean.csv"
test_out  = PROC / "test_clean.csv"

train_clean.to_csv(train_out, index=False)
test_clean.to_csv(test_out, index=False)

print("Saved:", train_out)
print("Saved:", test_out)


Saved: ..\data\processed\train_clean.csv
Saved: ..\data\processed\test_clean.csv


## Final Checks
Confirm the final column list and basic counts.


In [23]:
# Column order
print(train_clean.columns.tolist())

# Shapes check
print("train_clean:", train_clean.shape, "test_clean:", test_clean.shape)


['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'FamilySize', 'IsAlone', 'Fare', 'Embarked', 'Deck', 'HasCabin', 'TicketGroupSize', 'TicketPrefix', 'Title', 'PassengerId']
train_clean: (891, 16) test_clean: (418, 15)


In [24]:
import pandas as pd

train_clean = pd.read_csv("../data/processed/train_clean.csv")
print(train_clean.columns.tolist())

['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'FamilySize', 'IsAlone', 'Fare', 'Embarked', 'Deck', 'HasCabin', 'TicketGroupSize', 'TicketPrefix', 'Title', 'PassengerId']
